# Cell 1 — Install and imports

In [1]:
!pip -q install opencv-python-headless albumentations torchvision

import os
import gc
import json
import random
import shutil
import warnings
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

import albumentations as A
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score

warnings.filterwarnings("ignore")

print("Imports done")

Imports done


# Cell 2 — Paths and input loading

In [2]:
# Notebook 1 output dataset
PREP_ROOT = "/kaggle/input/datasets/naresh26032005/vinbigdata-prep"

CACHE_CSV = os.path.join(PREP_ROOT, "cache_df.csv")
SPLIT_DIR = os.path.join(PREP_ROOT, "splits")
CACHE_DIR = os.path.join(PREP_ROOT, "cache_640")

assert os.path.exists(PREP_ROOT), f"PREP_ROOT not found: {PREP_ROOT}"
assert os.path.exists(CACHE_CSV), "cache_df.csv not found"
assert os.path.exists(os.path.join(SPLIT_DIR, "train_split.csv")), "train_split.csv not found"
assert os.path.exists(os.path.join(SPLIT_DIR, "val_split.csv")), "val_split.csv not found"

cache_df = pd.read_csv(CACHE_CSV)
train_df = pd.read_csv(os.path.join(SPLIT_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(SPLIT_DIR, "val_split.csv"))

print("PREP_ROOT:", PREP_ROOT)
print("cache_df:", cache_df.shape)
print("train_df:", train_df.shape)
print("val_df:", val_df.shape)

print(train_df.head())

PREP_ROOT: /kaggle/input/datasets/naresh26032005/vinbigdata-prep
cache_df: (15000, 6)
train_df: (12000, 6)
val_df: (3000, 6)
                           image_id  \
0  000434271f63a053c4128a0ba6352c7f   
1  0005e8e3701dfb1dd93d53e2ff537b6e   
2  0006e0a85696f6bb578e84fafa9a5607   
3  0007d316f756b3fa0baea2ff514ce945   
4  000ae00eb3942d27e0b97903dd563a6e   

                                        src_path_rel  \
0  /kaggle/input/competitions/vinbigdata-chest-xr...   
1  /kaggle/input/competitions/vinbigdata-chest-xr...   
2  /kaggle/input/competitions/vinbigdata-chest-xr...   
3  /kaggle/input/competitions/vinbigdata-chest-xr...   
4  /kaggle/input/competitions/vinbigdata-chest-xr...   

                                   cache_path_rel  has_abnormal  \
0  cache_640/000434271f63a053c4128a0ba6352c7f.jpg             0   
1  cache_640/0005e8e3701dfb1dd93d53e2ff537b6e.jpg             1   
2  cache_640/0006e0a85696f6bb578e84fafa9a5607.jpg             0   
3  cache_640/0007d316f756b3fa0baea2

# Cell 3 — Config

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLASS_NAMES = [
    "Aortic enlargement",
    "Atelectasis",
    "Calcification",
    "Cardiomegaly",
    "Consolidation",
    "ILD",
    "Infiltration",
    "Lung Opacity",
    "Nodule/Mass",
    "Other lesion",
    "Pleural effusion",
    "Pleural thickening",
    "Pneumothorax",
    "Pulmonary fibrosis",
    "No finding",
]
NUM_CLASSES = len(CLASS_NAMES)
NO_FINDING_ID = 14
IMG_SIZE = 224

# Keep within Kaggle time budget
EPOCHS_BINARY = 10
EPOCHS_MULTI = 15
PATIENCE = 2

BATCH_BINARY = 32
BATCH_MULTI = 24
NUM_WORKERS = 2

WORK_DIR = "/kaggle/working/resnet50_advanced"
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR, ignore_errors=True)
os.makedirs(WORK_DIR, exist_ok=True)

print("Work dir:", WORK_DIR)

Device: cuda
Work dir: /kaggle/working/resnet50_advanced


# Cell 4 — Helpers and preprocessing

In [4]:
def ensure_uint8(img):
    img = img.astype(np.float32)
    img = img - img.min()
    mx = img.max()
    if mx > 0:
        img = img / mx
    return (img * 255.0).clip(0, 255).astype(np.uint8)

def chest_roi_crop(img, pad=0.06):
    """
    Conservative ROI crop to remove borders and focus on the chest region.
    """
    thr = max(5, int(np.percentile(img, 5)))
    mask = (img > thr).astype(np.uint8) * 255
    coords = cv2.findNonZero(mask)
    if coords is None:
        return img

    x, y, w, h = cv2.boundingRect(coords)
    H, W = img.shape[:2]

    x1 = max(0, int(x - pad * w))
    y1 = max(0, int(y - pad * h))
    x2 = min(W, int(x + w + pad * w))
    y2 = min(H, int(y + h + pad * h))

    crop = img[y1:y2, x1:x2]
    return crop if crop.size > 0 else img

def lung_window(img):
    lo = np.percentile(img, 2)
    hi = np.percentile(img, 70)
    out = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return (out * 255).astype(np.uint8)

def mediastinal_window(img):
    lo = np.percentile(img, 10)
    hi = np.percentile(img, 90)
    out = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return (out * 255).astype(np.uint8)

def detail_window(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(img)

def cxr_to_3window(img_gray):
    img_gray = ensure_uint8(img_gray)
    return np.stack(
        [
            lung_window(img_gray),
            mediastinal_window(img_gray),
            detail_window(img_gray),
        ],
        axis=-1,
    )

def read_gray_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Unable to read image: {path}")
    return img

def preprocess_resnet_image(path, img_size=224, use_roi=True):
    img = read_gray_image(path)
    img = ensure_uint8(img)
    if use_roi:
        img = chest_roi_crop(img)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    img3 = cxr_to_3window(img)

    tensor = torch.from_numpy(img3).permute(2, 0, 1).float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    tensor = (tensor - mean) / std
    return tensor

def preprocess_tta_views(path, img_size=224, use_roi=True):
    img = read_gray_image(path)
    img = ensure_uint8(img)
    if use_roi:
        img = chest_roi_crop(img)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return [img, cv2.flip(img, 1)]

print("Preprocessing ready")

Preprocessing ready


# Cell 5 — Augmentations

In [5]:
def build_train_aug():
    return A.Compose(
        [
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.03,
                scale_limit=0.08,
                rotate_limit=10,
                p=0.6,
                border_mode=cv2.BORDER_CONSTANT,
            ),
            A.RandomBrightnessContrast(p=0.35),
            A.GaussNoise(p=0.12),
            A.MotionBlur(p=0.08),
        ]
    )

def build_val_aug():
    return A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)])

train_aug = build_train_aug()
val_aug = build_val_aug()

print("Augmentations ready")

Augmentations ready


# Cell 6 — Parse labels and build sampling weights

In [6]:
def parse_classes(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [int(x) for x in str(s).split() if str(x).strip().isdigit()]

# class frequency from training split
class_freq = Counter()
for cls_str in train_df["classes"].tolist():
    for c in parse_classes(cls_str):
        if c != NO_FINDING_ID:
            class_freq[c] += 1

print("Class frequency:", class_freq)

def make_binary_sampler(df):
    labels = df["has_abnormal"].astype(int).tolist()
    counts = Counter(labels)
    weights = [1.0 / counts[int(y)] for y in labels]
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

def make_multilabel_sampler(df):
    """
    Weight images by rarity of classes present.
    Normal images get weight 1.
    """
    weights = []
    for _, row in df.iterrows():
        cls = [c for c in parse_classes(row["classes"]) if c != NO_FINDING_ID]
        if len(cls) == 0:
            weights.append(1.0)
        else:
            w = 0.0
            for c in cls:
                w += 1.0 / max(class_freq.get(c, 1), 1)
            weights.append(float(np.clip(w * 10.0, 1.0, 10.0)))
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

train_sampler_bin = make_binary_sampler(train_df)
train_sampler_ml = make_multilabel_sampler(train_df)

print("Samplers ready")

Class frequency: Counter({0: 2429, 3: 1823, 11: 1593, 13: 1290, 7: 1059, 9: 903, 10: 828, 8: 672, 6: 485, 2: 365, 5: 312, 4: 283, 1: 152, 12: 80})
Samplers ready


# Cell 7 — Datasets

In [7]:
class BinaryCXRDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.aug = train_aug if train else val_aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(PREP_ROOT, row["cache_path_rel"])
        img = read_gray_image(path)
        img = ensure_uint8(img)
        img = chest_roi_crop(img)

        img = self.aug(image=img)["image"]
        img3 = cxr_to_3window(img)

        x = torch.from_numpy(img3).permute(2, 0, 1).float() / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        x = (x - mean) / std

        y = torch.tensor(float(row["has_abnormal"]), dtype=torch.float32)
        return x, y

class MultiLabelCXRDataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train
        self.aug = train_aug if train else val_aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(PREP_ROOT, row["cache_path_rel"])
        img = read_gray_image(path)
        img = ensure_uint8(img)
        img = chest_roi_crop(img)

        img = self.aug(image=img)["image"]
        img3 = cxr_to_3window(img)

        x = torch.from_numpy(img3).permute(2, 0, 1).float() / 255.0
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        x = (x - mean) / std

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        cls_list = parse_classes(row["classes"])

        for c in cls_list:
            if c != NO_FINDING_ID:
                target[int(c)] = 1.0

        if int(row["has_no_finding"]) == 1:
            target[:] = 0.0
            target[NO_FINDING_ID] = 1.0

        return x, torch.tensor(target, dtype=torch.float32)

print("Datasets ready")

Datasets ready


# Cell 8 — Dataloaders

In [8]:
train_bin_ds = BinaryCXRDataset(train_df, train=True)
val_bin_ds = BinaryCXRDataset(val_df, train=False)

train_ml_ds = MultiLabelCXRDataset(train_df, train=True)
val_ml_ds = MultiLabelCXRDataset(val_df, train=False)

train_bin_loader = DataLoader(
    train_bin_ds,
    batch_size=BATCH_BINARY,
    sampler=train_sampler_bin,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

val_bin_loader = DataLoader(
    val_bin_ds,
    batch_size=BATCH_BINARY,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

train_ml_loader = DataLoader(
    train_ml_ds,
    batch_size=BATCH_MULTI,
    sampler=train_sampler_ml,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

val_ml_loader = DataLoader(
    val_ml_ds,
    batch_size=BATCH_MULTI,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

print("DataLoaders ready")

DataLoaders ready


# Cell 9 — Models and losses

In [9]:
class ResNet50Classifier(nn.Module):
    def __init__(self, out_dim=1, pretrained=True, dropout=0.25):
        super().__init__()
        try:
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            backbone = models.resnet50(weights=weights)
        except Exception:
            backbone = models.resnet50(pretrained=pretrained)

        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, out_dim),
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

class ResNet50Binary(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.model = ResNet50Classifier(out_dim=1, pretrained=pretrained)

    def forward(self, x):
        return self.model(x).squeeze(1)

class ResNet50MultiLabel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        self.model = ResNet50Classifier(out_dim=num_classes, pretrained=pretrained)

    def forward(self, x):
        return self.model(x)

class FocalLossBinary(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        targets = targets.float()
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        pt = torch.where(targets == 1, prob, 1 - prob)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * (1 - pt) ** self.gamma * bce
        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss

print("Models and losses ready")

Models and losses ready


# Cell 10 — Training and validation helpers

In [10]:
def train_one_epoch_binary(model, loader, optimizer, criterion, scaler=None):
    model.train()
    running_loss = 0.0
    all_logits = []
    all_targets = []

    for x, y in tqdm(loader, desc="train_binary", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(scaler is not None and device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if scaler is not None and device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * x.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(y.detach().cpu())

    return running_loss / len(loader.dataset), torch.cat(all_logits), torch.cat(all_targets)

def validate_binary(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="val_binary", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)

            running_loss += loss.item() * x.size(0)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())

    return running_loss / len(loader.dataset), torch.cat(all_logits), torch.cat(all_targets)

def train_one_epoch_multilabel(model, loader, optimizer, criterion, scaler=None):
    model.train()
    running_loss = 0.0
    all_logits = []
    all_targets = []

    for x, y in tqdm(loader, desc="train_multi", leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(scaler is not None and device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        if scaler is not None and device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        running_loss += loss.item() * x.size(0)
        all_logits.append(logits.detach().cpu())
        all_targets.append(y.detach().cpu())

    return running_loss / len(loader.dataset), torch.cat(all_logits), torch.cat(all_targets)

def validate_multilabel(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="val_multi", leave=False):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)
            loss = criterion(logits, y)

            running_loss += loss.item() * x.size(0)
            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())

    return running_loss / len(loader.dataset), torch.cat(all_logits), torch.cat(all_targets)

def tta_predict_binary(model, path):
    views = preprocess_tta_views(path, img_size=IMG_SIZE, use_roi=True)
    probs = []

    model.eval()
    with torch.no_grad():
        for v in views:
            img3 = cxr_to_3window(v)
            x = torch.from_numpy(img3).permute(2, 0, 1).float() / 255.0
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            x = ((x - mean) / std).unsqueeze(0).to(device)
            logit = model(x)
            probs.append(torch.sigmoid(logit).item())

    return float(np.mean(probs))

def tta_predict_multilabel(model, path):
    views = preprocess_tta_views(path, img_size=IMG_SIZE, use_roi=True)
    probs_list = []

    model.eval()
    with torch.no_grad():
        for v in views:
            img3 = cxr_to_3window(v)
            x = torch.from_numpy(img3).permute(2, 0, 1).float() / 255.0
            mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
            x = ((x - mean) / std).unsqueeze(0).to(device)
            logits = model(x)
            probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
            probs_list.append(probs)

    return np.mean(probs_list, axis=0)

def tune_per_class_thresholds(probs, targets, thresholds=None):
    if thresholds is None:
        thresholds = np.arange(0.05, 0.96, 0.05)

    best = np.full(probs.shape[1], 0.5, dtype=np.float32)
    best_scores = np.full(probs.shape[1], -1.0, dtype=np.float32)

    for c in range(probs.shape[1]):
        y_true = targets[:, c]
        if len(np.unique(y_true)) < 2:
            best[c] = 0.5
            best_scores[c] = 0.0
            continue

        for t in thresholds:
            pred = (probs[:, c] >= t).astype(int)
            score = f1_score(y_true, pred, zero_division=0)
            if score > best_scores[c]:
                best_scores[c] = score
                best[c] = t

    return best, best_scores

print("Training helpers ready")

Training helpers ready


# Cell 11 — Train binary model

In [11]:
num_pos = int(train_df["has_abnormal"].sum())
num_neg = int(len(train_df) - num_pos)
pos_weight_binary = torch.tensor([num_neg / max(num_pos, 1)], dtype=torch.float32, device=device)

binary_model = ResNet50Binary(pretrained=True).to(device)
optimizer_bin = torch.optim.AdamW(binary_model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion_bin = nn.BCEWithLogitsLoss(pos_weight=pos_weight_binary)
scaler_bin = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

best_bin_loss = float("inf")
best_bin_state = None
no_improve = 0

for epoch in range(1, EPOCHS_BINARY + 1):
    tr_loss, tr_logits, tr_targets = train_one_epoch_binary(
        binary_model, train_bin_loader, optimizer_bin, criterion_bin, scaler=scaler_bin
    )
    va_loss, va_logits, va_targets = validate_binary(binary_model, val_bin_loader, criterion_bin)

    va_probs = torch.sigmoid(va_logits).numpy()
    va_true = va_targets.numpy().astype(int)
    va_pred = (va_probs >= 0.5).astype(int)

    try:
        auc = roc_auc_score(va_true, va_probs)
    except Exception:
        auc = float("nan")

    f1 = f1_score(va_true, va_pred, zero_division=0)
    acc = accuracy_score(va_true, va_pred)

    print(
        f"[Binary Epoch {epoch}] "
        f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
        f"AUC={auc:.4f} F1@0.5={f1:.4f} ACC={acc:.4f}"
    )

    if va_loss < best_bin_loss:
        best_bin_loss = va_loss
        best_bin_state = {k: v.detach().cpu().clone() for k, v in binary_model.state_dict().items()}
        no_improve = 0
        print("✅ Binary improved")
    else:
        no_improve += 1
        print(f"⚠️ No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print("⛔ Early stopping binary")
            break

if best_bin_state is not None:
    binary_model.load_state_dict(best_bin_state)

binary_path = os.path.join(WORK_DIR, "resnet50_binary_best.pth")
torch.save(binary_model.state_dict(), binary_path)
print("Saved:", binary_path)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]


train_binary:   0%|          | 0/375 [00:00<?, ?it/s]

val_binary:   0%|          | 0/94 [00:00<?, ?it/s]

[Binary Epoch 1] train_loss=0.5881 val_loss=0.3136 AUC=0.9745 F1@0.5=0.8333 ACC=0.8883
✅ Binary improved


...

val_binary:   0%|          | 0/94 [00:00<?, ?it/s]

[Binary Epoch 8] train_loss=0.2017 val_loss=0.2540 AUC=0.9853 F1@0.5=0.9022 ACC=0.9413
⚠️ No improvement (1/2)


train_binary:   0%|          | 0/375 [00:00<?, ?it/s]

val_binary:   0%|          | 0/94 [00:00<?, ?it/s]

[Binary Epoch 9] train_loss=0.1890 val_loss=0.2703 AUC=0.9843 F1@0.5=0.9027 ACC=0.9417
⚠️ No improvement (2/2)
⛔ Early stopping binary
Saved: /kaggle/working/resnet50_advanced/resnet50_binary_best.pth


# Cell 12 — Train multilabel model

In [12]:
# class-wise positive weights
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, row in train_df.iterrows():
    cls = parse_classes(row["classes"])
    for c in cls:
        if c != NO_FINDING_ID:
            class_counts[int(c)] += 1

total_imgs = len(train_df)
pos_weight_ml = []
for c in range(NUM_CLASSES):
    if c == NO_FINDING_ID:
        pos_weight_ml.append(1.0)
    else:
        pos = max(class_counts[c], 1)
        neg = max(total_imgs - pos, 1)
        pos_weight_ml.append(neg / pos)

pos_weight_ml = torch.tensor(pos_weight_ml, dtype=torch.float32, device=device)

multi_model = ResNet50MultiLabel(pretrained=True).to(device)
optimizer_ml = torch.optim.AdamW(multi_model.parameters(), lr=1e-4, weight_decay=1e-4)
criterion_ml = nn.BCEWithLogitsLoss(pos_weight=pos_weight_ml)
scaler_ml = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

best_multi_loss = float("inf")
best_multi_state = None
no_improve = 0
best_global_threshold = 0.35
best_class_thresholds = np.full(NUM_CLASSES, 0.5, dtype=np.float32)

for epoch in range(1, EPOCHS_MULTI + 1):
    tr_loss, tr_logits, tr_targets = train_one_epoch_multilabel(
        multi_model, train_ml_loader, optimizer_ml, criterion_ml, scaler=scaler_ml
    )
    va_loss, va_logits, va_targets = validate_multilabel(multi_model, val_ml_loader, criterion_ml)

    va_probs = torch.sigmoid(va_logits).numpy()
    va_true = va_targets.numpy().astype(int)

    # tune a single global threshold for reporting
    global_scores = {}
    for t in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
        pred = (va_probs >= t).astype(int)
        global_scores[t] = f1_score(va_true, pred, average="micro", zero_division=0)

    best_global_threshold = max(global_scores, key=global_scores.get)
    va_pred_global = (va_probs >= best_global_threshold).astype(int)
    micro_f1 = f1_score(va_true, va_pred_global, average="micro", zero_division=0)
    macro_f1 = f1_score(va_true, va_pred_global, average="macro", zero_division=0)

    # per-class threshold tuning
    best_class_thresholds, per_class_scores = tune_per_class_thresholds(va_probs, va_true)

    print(
        f"[Multi Epoch {epoch}] "
        f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} "
        f"micro_f1={micro_f1:.4f} macro_f1={macro_f1:.4f} best_t={best_global_threshold}"
    )

    if va_loss < best_multi_loss:
        best_multi_loss = va_loss
        best_multi_state = {k: v.detach().cpu().clone() for k, v in multi_model.state_dict().items()}
        no_improve = 0
        print("✅ Multilabel improved")
    else:
        no_improve += 1
        print(f"⚠️ No improvement ({no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print("⛔ Early stopping multilabel")
            break

if best_multi_state is not None:
    multi_model.load_state_dict(best_multi_state)

multi_path = os.path.join(WORK_DIR, "resnet50_multilabel_best.pth")
torch.save(multi_model.state_dict(), multi_path)
np.save(os.path.join(WORK_DIR, "best_class_thresholds.npy"), best_class_thresholds)

print("Saved:", multi_path)
print("Best global threshold:", best_global_threshold)
print("Per-class thresholds saved")

train_multi:   0%|          | 0/500 [00:00<?, ?it/s]

val_multi:   0%|          | 0/125 [00:00<?, ?it/s]

[Multi Epoch 1] train_loss=0.9701 val_loss=0.7182 micro_f1=0.4724 macro_f1=0.3558 best_t=0.5
✅ Multilabel improved


train_multi:   0%|          | 0/500 [00:00<?, ?it/s]

val_multi:   0%|          | 0/125 [00:00<?, ?it/s]

...

val_multi:   0%|          | 0/125 [00:00<?, ?it/s]

[Multi Epoch 8] train_loss=0.5165 val_loss=0.5866 micro_f1=0.5992 macro_f1=0.4399 best_t=0.5
⚠️ No improvement (1/2)


train_multi:   0%|          | 0/500 [00:00<?, ?it/s]

val_multi:   0%|          | 0/125 [00:00<?, ?it/s]

[Multi Epoch 9] train_loss=0.5229 val_loss=0.5917 micro_f1=0.6377 macro_f1=0.4631 best_t=0.5
⚠️ No improvement (2/2)
⛔ Early stopping multilabel
Saved: /kaggle/working/resnet50_advanced/resnet50_multilabel_best.pth
Best global threshold: 0.5
Per-class thresholds saved


# Cell 13 — Final threshold tuning with TTA on validation

In [13]:
# Binary threshold tuning with TTA
val_binary_probs = []
val_binary_true = []

binary_model.eval()
with torch.no_grad():
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="TTA val binary"):
        path = os.path.join(PREP_ROOT, row["cache_path_rel"])
        prob = tta_predict_binary(binary_model, path)
        val_binary_probs.append(prob)
        val_binary_true.append(int(row["has_abnormal"]))

val_binary_probs = np.array(val_binary_probs)
val_binary_true = np.array(val_binary_true)

best_binary_threshold = 0.5
best_f1 = -1

for t in np.arange(0.05, 0.96, 0.05):
    pred = (val_binary_probs >= t).astype(int)
    f1 = f1_score(val_binary_true, pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_binary_threshold = t

print("Best binary threshold:", best_binary_threshold)
print("Binary F1:", best_f1)

# Multilabel threshold tuning with TTA
val_multi_probs = []
val_multi_true = []

multi_model.eval()
with torch.no_grad():
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="TTA val multilabel"):
        path = os.path.join(PREP_ROOT, row["cache_path_rel"])
        probs = tta_predict_multilabel(multi_model, path)
        val_multi_probs.append(probs)

        target = np.zeros(NUM_CLASSES, dtype=np.float32)
        cls_list = parse_classes(row["classes"])
        for c in cls_list:
            if c != NO_FINDING_ID:
                target[int(c)] = 1.0
        if int(row["has_no_finding"]) == 1:
            target[:] = 0.0
            target[NO_FINDING_ID] = 1.0

        val_multi_true.append(target)

val_multi_probs = np.vstack(val_multi_probs)
val_multi_true = np.vstack(val_multi_true).astype(int)

best_class_thresholds_tta, per_class_scores_tta = tune_per_class_thresholds(val_multi_probs, val_multi_true)

print("Updated class thresholds:", best_class_thresholds_tta)

# save final tuned thresholds
np.save(os.path.join(WORK_DIR, "best_class_thresholds_tta.npy"), best_class_thresholds_tta)
with open(os.path.join(WORK_DIR, "best_binary_threshold.txt"), "w") as f:
    f.write(str(best_binary_threshold))

# quick metrics
val_multi_pred = (val_multi_probs >= best_class_thresholds_tta).astype(int)
micro_f1 = f1_score(val_multi_true, val_multi_pred, average="micro", zero_division=0)
macro_f1 = f1_score(val_multi_true, val_multi_pred, average="macro", zero_division=0)

print("Multilabel micro F1:", micro_f1)
print("Multilabel macro F1:", macro_f1)

TTA val binary:   0%|          | 0/3000 [00:00<?, ?it/s]

Best binary threshold: 0.7000000000000001
Binary F1: 0.9208549971114962


TTA val multilabel:   0%|          | 0/3000 [00:00<?, ?it/s]

Updated class thresholds: [0.7  0.95 0.75 0.75 0.9  0.8  0.85 0.75 0.75 0.5  0.75 0.55 0.95 0.6
 0.55]
Multilabel micro F1: 0.7347599965896496
Multilabel macro F1: 0.5237347489984779


# Cell 14 — Save summary and check output size

In [14]:
summary = {
    "PREP_ROOT": PREP_ROOT,
    "binary_path": binary_path,
    "multi_path": multi_path,
    "best_binary_threshold": float(best_binary_threshold),
    "best_global_threshold_epoch": float(best_global_threshold),
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "binary_epochs": EPOCHS_BINARY,
    "multi_epochs": EPOCHS_MULTI,
}

with open(os.path.join(WORK_DIR, "resnet_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

total_bytes = 0
for root, _, files in os.walk(WORK_DIR):
    for f in files:
        total_bytes += os.path.getsize(os.path.join(root, f))

print(f"Approx output size: {total_bytes / (1024**3):.3f} GB")

{
  "PREP_ROOT": "/kaggle/input/datasets/naresh26032005/vinbigdata-prep",
  "binary_path": "/kaggle/working/resnet50_advanced/resnet50_binary_best.pth",
  "multi_path": "/kaggle/working/resnet50_advanced/resnet50_multilabel_best.pth",
  "best_binary_threshold": 0.7000000000000001,
  "best_global_threshold_epoch": 0.5,
  "num_classes": 15,
  "img_size": 224,
  "binary_epochs": 10,
  "multi_epochs": 15
}
Approx output size: 0.184 GB


# Cell 15 — Sanity check on one validation image

In [15]:
sample_row = val_df.iloc[0]
sample_path = os.path.join(PREP_ROOT, sample_row["cache_path_rel"])

print("Sample image:", sample_row["image_id"])

bin_prob = tta_predict_binary(binary_model, sample_path)
multi_probs = tta_predict_multilabel(multi_model, sample_path)

print("Binary probability:", bin_prob)
print("Top 5 multilabel probabilities:")
top5 = np.argsort(-multi_probs)[:5]
for i in top5:
    print(CLASS_NAMES[i], float(multi_probs[i]))

# display image
img = read_gray_image(sample_path)
plt.figure(figsize=(6, 6))
plt.imshow(img, cmap="gray")
plt.axis("off")
plt.show()

Sample image: 00053190460d56c53cc3e57321387478
Binary probability: 8.568173939238477e-06
Top 5 multilabel probabilities:
No finding 0.9999932646751404
Other lesion 4.433307549334131e-05
Pleural thickening 4.106396227143705e-05
Atelectasis 3.148753603454679e-05
Nodule/Mass 2.322043656022288e-05


NameError: name 'plt' is not defined